In [ ]:
import ee
import geemap.core as geemap
ee.Authenticate()
ee.Initialize(project='ee-kieranmcevoy07')

In [ ]:
import geemap
import ee

ee.Initialize(project='ee-kieranmcevoy07')

california = ee.FeatureCollection("TIGER/2018/States").filter(ee.Filter.eq('NAME', 'California'))

dataset = ee.ImageCollection("LANDSAT/LC09/C02/T1").filterDate('2025-01-05', '2025-01-21').filterBounds(california) # Pacific palisades fire 2025

def apply_scale_factors(image):
    optical_bands = image.select('B6')
    thermal_bands_b10 = image.select('B10')
    thermal_bands_b11 = image.select('B11')

    # Scaling each band between 0-1
    scaled_optical_bands = optical_bands.subtract(1).divide(65535 - 1)
    scaled_thermal_bands_b10 = thermal_bands_b10.subtract(1).divide(65535 - 1)
    scaled_thermal_bands_b11 = thermal_bands_b11.subtract(1).divide(65535 - 1)

    # Add the scaled bands to the image
    return image.addBands(scaled_optical_bands, None, True).addBands(scaled_thermal_bands_b10, None, True).addBands(scaled_thermal_bands_b11, None, True)

dataset = dataset.map(apply_scale_factors)


median_image = dataset.median()
band_names = median_image.bandNames()
print(f"Band names: {band_names.getInfo()}")


# Visualization
visualization = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 10000,
}

m = geemap.Map()
m.set_center(-119.4179, 36.7783, 6)  # Center map on California
m.addLayer(median_image, visualization, 'True Color (432)')
m.addLayer(california.style(color='red', fillColor='00000000'), {}, 'California Boundary')
m

Band names: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11', 'QA_PIXEL', 'QA_RADSAT', 'SAA', 'SZA', 'VAA', 'VZA']


Map(center=[36.7783, -119.4179], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Search…

In [ ]:
# Get the min and max values for B10
stats = median_image.select('B10').reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=california.geometry(),
    scale=10,
    bestEffort=True
)

# Print results
min_temp = stats.getNumber('B10_min').getInfo()
max_temp = stats.getNumber('B10_max').getInfo()

print(f"Thermal Band B10 Range: Min = {min_temp}, Max = {max_temp}")

Thermal Band B10 Range: Min = 0.09416486322879791, Max = 0.4277779459953308


In [ ]:
thermal_vis = {
    'bands': ['B10'],
    'min': 0,
    'max': 0.5,
    'palette': ['blue', 'cyan', 'yellow', 'orange', 'red']
}

m = geemap.Map()
m.set_center(-118.5255, 34.0484, 12)
m.addLayer(median_image, thermal_vis, 'Thermal Heatmap (B10)')

m.addLayer(california.style(color='black', fillColor='00000000'), {}, 'California Boundary')

m


Map(center=[34.0484, -118.5255], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Search…

In [ ]:
california = ee.FeatureCollection("TIGER/2018/States").filter(ee.Filter.eq('NAME', 'California'))

dataset = ee.ImageCollection("LANDSAT/LC09/C02/T1").filterDate('2024-10-26', '2024-10-27').filterBounds(california) # Pacific palisades fire 2025
#dataset = ee.ImageCollection("LANDSAT/LC09/C02/T1").filterDate('2024-12-15', '2024-12-31').filterBounds(california) # Before the fire
#dataset = ee.ImageCollection("LANDSAT/LC09/C02/T1").filterDate('2025-01-22', '2025-02-06').filterBounds(california) # Aftet the fire

def apply_scale_factors(image):
    optical_bands = image.select('B6')
    thermal_bands_b10 = image.select('B10')
    thermal_bands_b11 = image.select('B11')

    scaled_optical_bands = optical_bands.subtract(1).divide(65535 - 1)
    scaled_thermal_bands_b10 = thermal_bands_b10.subtract(1).divide(65535 - 1)
    scaled_thermal_bands_b11 = thermal_bands_b11.subtract(1).divide(65535 - 1)

    # Add the scaled bands to the image
    return image.addBands(scaled_optical_bands, None, True).addBands(scaled_thermal_bands_b10, None, True).addBands(scaled_thermal_bands_b11, None, True)

dataset = dataset.map(apply_scale_factors)


median_image = dataset.median()


# Visualization
visualization = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 10000,
}

stats = median_image.select('B10').reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=california.geometry(),
    scale=10,
    bestEffort=True
)

# Print results
min_temp = stats.getNumber('B10_min').getInfo()
max_temp = stats.getNumber('B10_max').getInfo()

thermal_vis = {
    'bands': ['B10'],
    'min': 0,
    'max': 0.48,
    'palette': ['blue','cyan','yellow', 'orange', 'red']
}






center = ee.Geometry.Point([-118.60, 34.09])

# rectangle for grid in degrees
height = 0.1
width = 2.5 * height

lon = ee.Number(center.coordinates().get(0))
lat = ee.Number(center.coordinates().get(1))

# bounding coordinates.
xmin = lon.subtract(width / 2)
xmax = lon.add(width / 2)
ymin = lat.subtract(height / 2)
ymax = lat.add(height / 2)

rectangle = ee.Geometry.Rectangle([xmin, ymin, xmax, ymax])
region = rectangle
# Generate grid
grid = ee.FeatureCollection(region.coveringGrid('EPSG:4326', scale=2500))
# Get grid size
grid_size = grid.size().getInfo()

# Print grid info
print(f"Number of grid cells: {grid_size}")


m = geemap.Map()
m.set_center(-118.60, 34.09, 12)  # Center map on palisades
m.addLayer(median_image, visualization, 'True Color (432)')
m.addLayer(median_image, thermal_vis, 'Thermal Heatmap (B10)')
m.addLayer(california.style(color='red', fillColor='00000000'), {}, 'California Boundary')
m.addLayer(grid.style(color='black', fillColor='00000000', width=2), {}, 'Grid Tiles')
m


Number of grid cells: 72


Map(center=[34.09, -118.6], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataG…

In [ ]:
#Tile generator
#Satilltes return time = 16 days so each dataset takes 16 days
fire_dataset = ee.ImageCollection("LANDSAT/LC09/C02/T1").filterDate('2025-01-14', '2025-01-15').filterBounds(california) # Closest date avaliable after fire starts for Pacific palisades fire 2025
before_dataset = ee.ImageCollection("LANDSAT/LC09/C02/T1").filterDate('2024-12-29', '2024-12-30').filterBounds(california) # Before the fire
after_dataset = ee.ImageCollection("LANDSAT/LC09/C02/T1").filterDate('2025-01-30', '2025-01-31').filterBounds(california) # Aftet the fire
nofire_dataset = ee.ImageCollection("LANDSAT/LC09/C02/T1").filterDate('2024-10-26', '2024-10-27').filterBounds(california)

In [ ]:
import ee



# Define datasets
california = ee.Geometry.Point([-118.60, 34.09])  # Adjust this to your fire location

datasets = {
    "Fire": ee.ImageCollection("LANDSAT/LC09/C02/T1")
        .filterDate('2025-01-14', '2025-01-15')
        .filterBounds(california),
    "Before_Fire": ee.ImageCollection("LANDSAT/LC09/C02/T1")
        .filterDate('2024-12-29', '2024-12-30')
        .filterBounds(california),
    "After_Fire": ee.ImageCollection("LANDSAT/LC09/C02/T1")
        .filterDate('2025-01-30', '2025-01-31')
        .filterBounds(california),
    "No_Fire": ee.ImageCollection("LANDSAT/LC09/C02/T1")
        .filterDate('2024-10-26', '2024-10-27')
        .filterBounds(california)
}

# Define grid parameters
center = ee.Geometry.Point([-118.60, 34.09])

# Rectangle for grid in degrees
height = 0.1
width = 2.5 * height

lon = ee.Number(center.coordinates().get(0))
lat = ee.Number(center.coordinates().get(1))

# Bounding coordinates
xmin = lon.subtract(width / 2)
xmax = lon.add(width / 2)
ymin = lat.subtract(height / 2)
ymax = lat.add(height / 2)

rectangle = ee.Geometry.Rectangle([xmin, ymin, xmax, ymax])
region = rectangle

# Generate grid
grid = ee.FeatureCollection(region.coveringGrid('EPSG:4326', scale=2500))

# Convert to list for iteration
grid_list = grid.toList(grid.size())
grid_size = grid.size().getInfo()

# Loop through datasets and export tiles
for dataset_name, dataset in datasets.items():
    for i in range(1, grid_size+1):
        region_tile = ee.Feature(grid_list.get(i)).geometry()
        clipped_image = dataset.median()  # Get median image of dataset

        task = ee.batch.Export.image.toDrive(
            image=clipped_image,
            description=f'{dataset_name}_Tile{i}',
            folder=dataset_name,
            region=region_tile,
            scale=30,  # Meters per pixel
            maxPixels=1e10
        )
        print(f"Exporting {dataset_name} tile {i}...")
        task.start()


Exporting Fire tile 1...
Exporting Fire tile 2...
Exporting Fire tile 3...
Exporting Fire tile 4...
Exporting Fire tile 5...
Exporting Fire tile 6...
Exporting Fire tile 7...
Exporting Fire tile 8...
Exporting Fire tile 9...
Exporting Fire tile 10...
Exporting Fire tile 11...
Exporting Fire tile 12...
Exporting Fire tile 13...
Exporting Fire tile 14...
Exporting Fire tile 15...
Exporting Fire tile 16...
Exporting Fire tile 17...
Exporting Fire tile 18...
Exporting Fire tile 19...
Exporting Fire tile 20...
Exporting Fire tile 21...
Exporting Fire tile 22...
Exporting Fire tile 23...
Exporting Fire tile 24...
Exporting Fire tile 25...
Exporting Fire tile 26...
Exporting Fire tile 27...
Exporting Fire tile 28...
Exporting Fire tile 29...
Exporting Fire tile 30...
Exporting Fire tile 31...
Exporting Fire tile 32...
Exporting Fire tile 33...
Exporting Fire tile 34...
Exporting Fire tile 35...
Exporting Fire tile 36...
Exporting Fire tile 37...
Exporting Fire tile 38...
Exporting Fire tile 3